# Benchmarking

This benchmarking compares the model variants used in the SATNAC 2026 results (see the
companion [results.ipynb](./results.ipynb)) — as opposed to the slowfast-based
MViTv2\_\*\_16x4/\_32x3 variants benchmarked in
[sacair_2026/benchmark.ipynb](../sacair_2026/benchmark.ipynb) and the full PyTorch-native
model family benchmarked in [satnac_2025/benchmark.ipynb](../satnac_2025/benchmark.ipynb):
- S3D
- MViTv2_S
- MViTv2_S_e

S3D is benchmarked at both 16 and 32 frames, since it is run at both in the paper's
"Also at 32 frames" variation. MViTv2_S is benchmarked at 16 frames, matching the currently
submitted draft's spec; its interpolated-positional-encoding variant, MViTv2_S_e, uses 32
frames, swept over batch size until each model/frame-count/mode combination OOMs -- the
largest batch size shown per combination is therefore the last one that still fit.

During training, a stable GPU clock speed range of 1910 - 1930 MHz was recorded, so the GPU
clock speed is locked with a slight margin at 1900 with the command:
```bash
sudo nvidia-smi -lgc 1900
```
and can be reset with:
```bash
sudo nvidia-smi -rgc
```

The benchmarking script used was: [src.benchmark.py](../../benchmark.py).
The benchmarks are loaded from: [all_benchmark.json](../all_benchmark.json).

In [1]:
import json

import pandas as pd

from src.run_types import RESULTS_DIR


In [2]:
def load_satnac_2026_benchmarks(benchmark_path):
    """
    Load the shared all_benchmark.json, keep only the model variants used in the SATNAC 2026
    results (S3D, MViTv2_S, MViTv2_S_e), drop OOM runs, map model names, and return separate
    DataFrames for training and inference.
    """
    with open(benchmark_path, "r") as f:
        raw = json.load(f)

    runs = raw["runs"]

    # Model name mapping
    model_name_map = {
        "S3D": "S3D",
        "MViTv2_S": "MViTv2\\_S",
        "MViTv2_S_e": "MViTv2\\_S\\_e",
    }

    records = []

    for run in runs.values():
        arch = run.get("arch", "")
        if arch not in model_name_map:
            continue
        if "error" in run:
            continue

        config = run["config"]
        results = run["results"]

        records.append({
            "model": model_name_map[arch],  # mapped name
            "num_frames": config["num_frames"],
            "mode": "Train" if config.get("full_step", False) else "Infer",
            "batch_size": config["batch_size"],

            "gpu_util_mean": results["gpu_utilisation_percent"]["mean"],
            "gpu_util_std": results["gpu_utilisation_percent"]["std"],

            "latency_ms_mean": results["latency_ms"]["mean"],
            "latency_ms_std": results["latency_ms"]["std"],

            "throughput_samp_per_s_mean": results["throughput_samples_per_s"]["mean"],
            "throughput_samp_per_s_std": results["throughput_samples_per_s"]["std"],

            "peak_mem_mb_mean": results["peak_memory_mb"]["mean"],
            "peak_mem_mb_std": results["peak_memory_mb"]["std"],
        })

    df = pd.DataFrame(records)

    metric_cols = [
        "gpu_util_mean", "gpu_util_std",
        "latency_ms_mean", "latency_ms_std",
        "throughput_samp_per_s_mean", "throughput_samp_per_s_std",
        "peak_mem_mb_mean", "peak_mem_mb_std",
    ]
    df[metric_cols] = df[metric_cols].round(2)

    # Sort by model, then num_frames, then batch_size
    train_df = df[df["mode"] == "Train"].sort_values(
        ["model", "num_frames", "batch_size"]
    ).reset_index(drop=True)
    infer_df = df[df["mode"] == "Infer"].sort_values(
        ["model", "num_frames", "batch_size"]
    ).reset_index(drop=True)

    return train_df, infer_df


## Load and display results

In [3]:
train_df, infer_df = load_satnac_2026_benchmarks(RESULTS_DIR / "all_benchmark.json")

print("Training DataFrame:")
display(train_df)

print("\nInference DataFrame:")
display(infer_df)


Training DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S,16,Train,1,98.66,0.02,175.50,0.39,5.70,0.01,1802.01,0.0
1,MViTv2\_S,16,Train,2,99.36,0.01,336.64,0.56,5.94,0.01,3294.64,0.0
2,MViTv2\_S,16,Train,4,99.79,0.02,643.22,0.72,6.22,0.01,6224.38,0.0
3,MViTv2\_S\_e,32,Train,1,99.49,0.01,427.50,0.65,2.34,0.00,3947.98,0.0
4,MViTv2\_S\_e,32,Train,2,99.87,0.01,837.06,0.92,2.39,0.00,7569.27,0.0
5,S3D,16,Train,1,99.97,0.07,31.46,0.00,31.79,0.00,546.84,0.0
6,S3D,16,Train,2,99.97,0.05,58.22,0.00,34.35,0.00,1012.14,0.0
7,S3D,16,Train,4,99.99,0.02,109.85,0.01,36.41,0.00,1945.76,0.0
8,S3D,16,Train,8,100.00,0.01,214.41,0.01,37.31,0.00,3821.83,0.0
9,S3D,16,Train,16,100.00,0.00,421.49,0.02,37.96,0.00,7568.14,0.0



Inference DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S,16,Infer,1,98.00,0.00,50.75,0.05,19.70,0.02,417.83,0.0
1,MViTv2\_S,16,Infer,2,99.00,0.00,97.85,0.17,20.44,0.04,680.53,0.0
2,MViTv2\_S,16,Infer,4,99.65,0.05,192.34,0.07,20.80,0.01,1212.29,0.0
3,MViTv2\_S,16,Infer,8,100.00,0.00,377.86,0.27,21.17,0.02,2277.22,0.0
4,MViTv2\_S,16,Infer,16,100.00,0.00,750.87,0.97,21.31,0.03,4406.08,0.0
5,MViTv2\_S\_e,32,Infer,1,99.01,0.02,127.40,0.25,7.85,0.02,1134.06,0.0
6,MViTv2\_S\_e,32,Infer,2,99.84,0.04,252.48,0.13,7.92,0.00,2114.95,0.0
7,MViTv2\_S\_e,32,Infer,4,100.00,0.00,497.04,0.09,8.05,0.00,4081.24,0.0
8,MViTv2\_S\_e,32,Infer,8,100.00,0.00,986.99,0.07,8.11,0.00,8015.75,0.0
9,S3D,16,Infer,1,100.00,0.00,9.86,0.00,101.44,0.02,163.19,0.0


### Prep for LaTeX

In [4]:
# Drop standard deviation columns and rename for table output
mean_cols = {
    "model": "Model",
    "num_frames": "Frames",
    "batch_size": "BS",
    "gpu_util_mean": "GPU Util.",
    "latency_ms_mean": "Latency (ms)",
    "throughput_samp_per_s_mean": "Throughput (samp/s)",
    "peak_mem_mb_mean": "Peak Mem. (MB)",
}

train_mean_df = train_df[list(mean_cols.keys())].rename(columns=mean_cols)
infer_mean_df = infer_df[list(mean_cols.keys())].rename(columns=mean_cols)

def format_benchmark_df(df):
    df = df.copy()
    df["GPU Util."] = df["GPU Util."].apply(lambda x: f"{x:.2f}\\%")
    df["Latency (ms)"] = df["Latency (ms)"].apply(lambda x: f"{x:,.2f}")
    df["Throughput (samp/s)"] = df["Throughput (samp/s)"].apply(lambda x: f"{x:,.2f}")
    df["Peak Mem. (MB)"] = df["Peak Mem. (MB)"].apply(lambda x: f"{x:,.2f}")
    return df

train_mean_df = format_benchmark_df(train_mean_df)
infer_mean_df = format_benchmark_df(infer_mean_df)

display(train_mean_df)
display(infer_mean_df)


,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S,16,1,98.66\%,175.50,5.70,"1,802.01"
1,MViTv2\_S,16,2,99.36\%,336.64,5.94,"3,294.64"
2,MViTv2\_S,16,4,99.79\%,643.22,6.22,"6,224.38"
3,MViTv2\_S\_e,32,1,99.49\%,427.50,2.34,"3,947.98"
4,MViTv2\_S\_e,32,2,99.87\%,837.06,2.39,"7,569.27"
5,S3D,16,1,99.97\%,31.46,31.79,546.84
6,S3D,16,2,99.97\%,58.22,34.35,"1,012.14"
7,S3D,16,4,99.99\%,109.85,36.41,"1,945.76"
8,S3D,16,8,100.00\%,214.41,37.31,"3,821.83"
9,S3D,16,16,100.00\%,421.49,37.96,"7,568.14"


,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S,16,1,98.00\%,50.75,19.70,417.83
1,MViTv2\_S,16,2,99.00\%,97.85,20.44,680.53
2,MViTv2\_S,16,4,99.65\%,192.34,20.80,"1,212.29"
3,MViTv2\_S,16,8,100.00\%,377.86,21.17,"2,277.22"
4,MViTv2\_S,16,16,100.00\%,750.87,21.31,"4,406.08"
5,MViTv2\_S\_e,32,1,99.01\%,127.40,7.85,"1,134.06"
6,MViTv2\_S\_e,32,2,99.84\%,252.48,7.92,"2,114.95"
7,MViTv2\_S\_e,32,4,100.00\%,497.04,8.05,"4,081.24"
8,MViTv2\_S\_e,32,8,100.00\%,986.99,8.11,"8,015.75"
9,S3D,16,1,100.00\%,9.86,101.44,163.19


### Print LaTeX

In [5]:
def fixhlines(txt: str) -> str:
    return (
        txt.replace("\\toprule", "\\hline")
        .replace("\\midrule", "\\hline")
        .replace("\\bottomrule", "\\hline")
    )


In [6]:
train_latex = train_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Training benchmark results for the SATNAC 2026 model variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:satnac_2026_benchmark_train",
    position="ht",
)

infer_latex = infer_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Inference benchmark results for the SATNAC 2026 model variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:satnac_2026_benchmark_infer",
    position="ht",
)

print("Training Table LaTeX:")
print(fixhlines(train_latex))
print("\nInference Table LaTeX:")
print(fixhlines(infer_latex))


Training Table LaTeX:
\begin{table}[ht]
\caption{Training benchmark results for the SATNAC 2026 model variants on an NVIDIA RTX 3060 12GB GPU.}
\label{tab:satnac_2026_benchmark_train}
\begin{tabular}{|l|c|c|r|r|r|r|}
\hline
Model & Frames & BS & GPU Util. & Latency (ms) & Throughput (samp/s) & Peak Mem. (MB) \\
\hline
MViTv2\_S & 16 & 1 & 98.66\% & 175.50 & 5.70 & 1,802.01 \\
MViTv2\_S & 16 & 2 & 99.36\% & 336.64 & 5.94 & 3,294.64 \\
MViTv2\_S & 16 & 4 & 99.79\% & 643.22 & 6.22 & 6,224.38 \\
MViTv2\_S\_e & 32 & 1 & 99.49\% & 427.50 & 2.34 & 3,947.98 \\
MViTv2\_S\_e & 32 & 2 & 99.87\% & 837.06 & 2.39 & 7,569.27 \\
S3D & 16 & 1 & 99.97\% & 31.46 & 31.79 & 546.84 \\
S3D & 16 & 2 & 99.97\% & 58.22 & 34.35 & 1,012.14 \\
S3D & 16 & 4 & 99.99\% & 109.85 & 36.41 & 1,945.76 \\
S3D & 16 & 8 & 100.00\% & 214.41 & 37.31 & 3,821.83 \\
S3D & 16 & 16 & 100.00\% & 421.49 & 37.96 & 7,568.14 \\
S3D & 32 & 1 & 99.97\% & 57.87 & 17.28 & 1,010.27 \\
S3D & 32 & 2 & 99.99\% & 110.81 & 18.05 & 1,945.76 \\
S3D